# Enhanced Domain-DSL Query Engine
## Testing Hierarchical OG-RAG with Complex Ontologies

This notebook tests the new engine designed for complex, hierarchical ontologies:
- Domain: 24 classes with 56 nested properties
- DSL: 21 entities categorized by type and kind
- Multi-level retrieval: Classes → Properties + DSL entities

In [1]:
import sys
sys.path.append('/media/thuongnv/New Volume/Code/Github/ograg2-1')

import yaml
import warnings
warnings.filterwarnings('ignore')

## 1. Load API Keys

In [2]:
# Load API keys
with open('api_keys.yaml', 'r') as f:
    api_keys = yaml.safe_load(f)

groq_api_key = api_keys['GROQ_API_KEY']
print("✓ API keys loaded")

✓ API keys loaded


## 2. Initialize LLM and Embeddings

In [4]:
from langchain_groq import ChatGroq
from langchain_community.embeddings import HuggingFaceEmbeddings

# Initialize LLM
llm = ChatGroq(
    model="llama-3.3-70b-versatile",
    api_key=groq_api_key,
    temperature=0
)
print("✓ LLM initialized")

# Initialize embeddings
embed_model = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)
print("✓ Embeddings initialized")

✓ LLM initialized
✓ Embeddings initialized
✓ Embeddings initialized


## 3. Initialize Enhanced Engine

In [5]:
# Import directly from file to avoid __init__.py issues
import importlib.util
spec = importlib.util.spec_from_file_location(
    "enhanced_engine", 
    "/media/thuongnv/New Volume/Code/Github/ograg2-1/query_engine/enhanced_domain_dsl_engine.py"
)
enhanced_engine = importlib.util.module_from_spec(spec)
spec.loader.exec_module(enhanced_engine)

EnhancedDomainDSLQueryEngine = enhanced_engine.EnhancedDomainDSLQueryEngine

engine = EnhancedDomainDSLQueryEngine(
    domain_ontology_path="data/kg/domain_extended/ontology/domain_extended.jsonld",
    dsl_ontology_path="data/kg/dsl_extended/ontology/dsl_extended.jsonld",
    llm=llm,
    embed_model=embed_model
)

Initializing Enhanced Domain-DSL Query Engine
Hierarchical OG-RAG for Complex Ontologies

📚 Parsing ontologies...
  ✓ Parsed 24 domain classes
    with 56 total properties
  ✓ Parsed 20 DSL entities:
    - DSLFunction: 9
    - DSLClause: 4
    - DSLModifier: 4
    - DSLQueryType: 3

🔨 Building domain hypergraph...
  Computing embeddings for 80 texts...
  ✓ Built graph with 24 classes
    Total properties: 56

🔨 Building DSL hypergraph...
  Computing embeddings for 20 DSL entities...
  ✓ Built graph with 24 classes
    Total properties: 56

🔨 Building DSL hypergraph...
  Computing embeddings for 20 DSL entities...
  ✓ Built graph with 20 DSL entities
    Organized by 4 types and 14 kinds

✓ Engine initialized successfully!
  ✓ Built graph with 20 DSL entities
    Organized by 4 types and 14 kinds

✓ Engine initialized successfully!


In [6]:
# Reload the module to get updated code
import importlib
spec = importlib.util.spec_from_file_location(
    "enhanced_engine", 
    "/media/thuongnv/New Volume/Code/Github/ograg2-1/query_engine/enhanced_domain_dsl_engine.py"
)
enhanced_engine = importlib.util.module_from_spec(spec)
spec.loader.exec_module(enhanced_engine)

EnhancedDomainDSLQueryEngine = enhanced_engine.EnhancedDomainDSLQueryEngine

# Recreate engine with updated code
engine = EnhancedDomainDSLQueryEngine(
    domain_ontology_path="data/kg/domain_extended/ontology/domain_extended.jsonld",
    dsl_ontology_path="data/kg/dsl_extended/ontology/dsl_extended.jsonld",
    llm=llm,
    embed_model=embed_model
)
print("\n✓ Engine reloaded with improved prompt")

Initializing Enhanced Domain-DSL Query Engine
Hierarchical OG-RAG for Complex Ontologies

📚 Parsing ontologies...
  ✓ Parsed 24 domain classes
    with 56 total properties
  ✓ Parsed 20 DSL entities:
    - DSLFunction: 9
    - DSLClause: 4
    - DSLModifier: 4
    - DSLQueryType: 3

🔨 Building domain hypergraph...
  Computing embeddings for 80 texts...
  ✓ Built graph with 24 classes
    Total properties: 56

🔨 Building DSL hypergraph...
  Computing embeddings for 20 DSL entities...
  ✓ Built graph with 24 classes
    Total properties: 56

🔨 Building DSL hypergraph...
  Computing embeddings for 20 DSL entities...
  ✓ Built graph with 20 DSL entities
    Organized by 4 types and 14 kinds

✓ Engine initialized successfully!

✓ Engine reloaded with improved prompt
  ✓ Built graph with 20 DSL entities
    Organized by 4 types and 14 kinds

✓ Engine initialized successfully!

✓ Engine reloaded with improved prompt


## 4. Test Queries

### Test Case 1: Simple Selection (CHONJ_KW)

In [7]:
query1 = "Find all users"
print(f"Query: {query1}")
print("="*70)

result1, ctx = engine.generate(query1)
print("Context: ")
print(ctx)
print(f"\nGenerated DSL:\n{result1}")

Query: Find all users
Context: 
=== DOMAIN KNOWLEDGE ===

Class: :Person
  Label: Person
  Description: A human user in the system. Typical questions: 'list all users', 'average age of users', 'users who live in the same city', 'friends of a person'.
  Examples: ex:alice, ex:bob
  Related Properties:
    - :hasEmail
      Label: email
      Description: The email address of a person or customer. Example questions: 'find emails of all customers', 'list users with missing email (OPTIONAL)'.
      Range: xsd:string
      Example: ex:alice :hasEmail "alice@example.com" .
    - :livesIn
      Label: lives in
      Description: Links a person to the city they live in. Typical questions: 'users who live in London', 'users living in the same city', 'number of users per city'.
      Range: :City
      Example: ex:alice :livesIn ex:London .
    - :hasName
      Label: name
      Description: The name of a person, customer, employee, or product. Example questions: 'list user names', 'names of all

### Test Case 2: Filter with Comparison (LOCJ_RR)

In [8]:
query2 = "Find users older than 18"
print(f"Query: {query2}")
print("="*70)

result2 = engine.generate(query2)
print(f"\nGenerated DSL:\n{result2}")

Query: Find users older than 18

Generated DSL:
('LOCJ_RR(?age > 18)', '=== DOMAIN KNOWLEDGE ===\n\nClass: :Person\n  Label: Person\n  Description: A human user in the system. Typical questions: \'list all users\', \'average age of users\', \'users who live in the same city\', \'friends of a person\'.\n  Examples: ex:alice, ex:bob\n  Related Properties:\n    - :hasAge\n      Label: age\n      Description: The age of a person in years. Typical questions: \'users older than 18\', \'average age of users\', \'youngest or oldest user age\'.\n      Range: xsd:integer\n      Example: ex:alice :hasAge 25 .\n    - :livesIn\n      Label: lives in\n      Description: Links a person to the city they live in. Typical questions: \'users who live in London\', \'users living in the same city\', \'number of users per city\'.\n      Range: :City\n      Example: ex:alice :livesIn ex:London .\n    - :hasEmail\n      Label: email\n      Description: The email address of a person or customer. Example questi

### Test Case 3: Aggregate Function (TRUNGG_BINH_PP)

In [9]:
query3 = "Calculate average age of users"
print(f"Query: {query3}")
print("="*70)

result3 = engine.generate(query3)
print(f"\nGenerated DSL:\n{result3}")

Query: Calculate average age of users

Generated DSL:
('TRUNGG_BINH_PP(?age)', '=== DOMAIN KNOWLEDGE ===\n\nClass: :Person\n  Label: Person\n  Description: A human user in the system. Typical questions: \'list all users\', \'average age of users\', \'users who live in the same city\', \'friends of a person\'.\n  Examples: ex:alice, ex:bob\n  Related Properties:\n    - :hasAge\n      Label: age\n      Description: The age of a person in years. Typical questions: \'users older than 18\', \'average age of users\', \'youngest or oldest user age\'.\n      Range: xsd:integer\n      Example: ex:alice :hasAge 25 .\n    - :livesIn\n      Label: lives in\n      Description: Links a person to the city they live in. Typical questions: \'users who live in London\', \'users living in the same city\', \'number of users per city\'.\n      Range: :City\n      Example: ex:alice :livesIn ex:London .\n    - :hasPhone\n      Label: phone number\n      Description: The phone number of a person, customer, or

### Test Case 4: Aggregate + Filter (Complex Query)

In [10]:
query4 = "Find average age of users older than 18"
print(f"Query: {query4}")
print("="*70)

result4 = engine.generate(query4)
print(f"\nGenerated DSL:\n{result4}")

Query: Find average age of users older than 18

Generated DSL:
('TRUNGG_BINH_PP(?age) LOCJ_RR(?age > 18)', '=== DOMAIN KNOWLEDGE ===\n\nClass: :Person\n  Label: Person\n  Description: A human user in the system. Typical questions: \'list all users\', \'average age of users\', \'users who live in the same city\', \'friends of a person\'.\n  Examples: ex:alice, ex:bob\n  Related Properties:\n    - :hasAge\n      Label: age\n      Description: The age of a person in years. Typical questions: \'users older than 18\', \'average age of users\', \'youngest or oldest user age\'.\n      Range: xsd:integer\n      Example: ex:alice :hasAge 25 .\n    - :livesIn\n      Label: lives in\n      Description: Links a person to the city they live in. Typical questions: \'users who live in London\', \'users living in the same city\', \'number of users per city\'.\n      Range: :City\n      Example: ex:alice :livesIn ex:London .\n    - :hasEmail\n      Label: email\n      Description: The email address of 

### Test Case 5: CONSTRUCT Query (XAY_DUNGWJ)
**This was the failing case in old engine!**

In [11]:
query5 = "Create friendship relations between people who live in the same city"
print(f"Query: {query5}")
print("="*70)

result5 = engine.generate(query5)
print(f"\nGenerated DSL:\n{result5}")

Query: Create friendship relations between people who live in the same city

Generated DSL:
('XAY_DUNGWJ { ?p1 :friendOf ?p2 } NOII_MA_KK { ?p1 :livesIn ?city . ?p2 :livesIn ?city . LOCJ_RR(?p1 != ?p2) }', '=== DOMAIN KNOWLEDGE ===\n\nClass: :City\n  Label: City\n  Description: A city where people live. Typical questions: \'users who live in London\', \'number of users per city\', \'friends living in the same city\'.\n  Examples: ex:London, ex:Paris\n  Related Properties:\n    - :cityInCountry\n      Label: city in country\n      Description: Links a city to its country. Typical questions: \'cities in each country\', \'orders grouped by country via the city of the customer\'.\n      Range: :Country\n      Example: ex:London :cityInCountry ex:UK .\n\nClass: :Person\n  Label: Person\n  Description: A human user in the system. Typical questions: \'list all users\', \'average age of users\', \'users who live in the same city\', \'friends of a person\'.\n  Examples: ex:alice, ex:bob\n  Rela

### Test Case 6: Group By with Count (NHOMM_THEO + DEMM)

In [12]:
query6 = "Count products in each category"
print(f"Query: {query6}")
print("="*70)

result6 = engine.generate(query6)
print(f"\nGenerated DSL:\n{result6}")

Query: Count products in each category

Generated DSL:
('DEMM_JJ(?product) NHOMM_THEO_YY ?category ?product :inCategory ?category .', '=== DOMAIN KNOWLEDGE ===\n\nClass: :Category\n  Label: Category\n  Description: A product category, such as \'Electronics\', \'Books\', or \'Clothing\'. Typical questions: \'products in the Electronics category\', \'number of products per category\'.\n  Examples: ex:Electronics, ex:Books\n\nClass: :Product\n  Label: Product\n  Description: A product that can be purchased in the system. Typical questions: \'products with price greater than 100\', \'average price per category\', \'top 10 most expensive products\'.\n  Examples: ex:product1, ex:product2\n  Related Properties:\n    - :inCategory\n      Label: in category\n      Description: Links a product to its category. Typical questions: \'products in the Electronics category\', \'number of products per category\'.\n      Range: :Category\n      Example: ex:product1 :inCategory ex:Electronics .\n    - :t

## 5. Inspect Retrieved Context
Let's examine what the engine retrieves for each query

In [13]:
# Test retrieval for CONSTRUCT query
test_query = "Create friendship relations between people who live in the same city"

print(f"Inspecting retrieval for: {test_query}")
print("="*70)

context = engine.retrieve_hierarchical_context(test_query)

print("\n📊 DOMAIN CLASSES RETRIEVED:")
for item in context['domain']:
    cls = item['class']
    print(f"\n  Class: {cls.identifier}")
    print(f"  Score: {item['class_score']:.4f}")
    print(f"  Properties: {len(item['properties'])}")
    for prop, score in item['properties'][:3]:  # Show top 3
        print(f"    - {prop.get('property')} (score: {score:.4f})")

print("\n\n📊 DSL ENTITIES RETRIEVED:")
for node, score in context['dsl']:
    print(f"\n  {node.node_type}: {node.keyword}")
    print(f"  Score: {score:.4f}")
    print(f"  Kind: {node.kind}")

Inspecting retrieval for: Create friendship relations between people who live in the same city

📊 DOMAIN CLASSES RETRIEVED:

  Class: :City
  Score: 0.5628
  Properties: 1
    - :cityInCountry (score: 0.3366)

  Class: :Person
  Score: 0.3611
  Properties: 5
    - :friendOf (score: 0.6471)
    - :livesIn (score: 0.5079)
    - :hasAddress (score: 0.2156)

  Class: :Country
  Score: 0.1994
  Properties: 0


📊 DSL ENTITIES RETRIEVED:

  DSLModifier: KHAC_NHAU_PP
  Score: 0.2847
  Kind: distinct

  DSLClause: NHOMM_THEO_YY
  Score: 0.2330
  Kind: group_by

  DSLClause: DIEU_KIEN_NHOMM_PP
  Score: 0.2002
  Kind: having

  DSLClause: LOCJ_RR
  Score: 0.1757
  Kind: filter

  DSLQueryType: 
  Score: 0.1204
  Kind: construct


## 6. Compare with Old Engine
Show differences in retrieved context

## 7. Performance Analysis

In [14]:
import time

test_queries = [
    "Find all users",
    "Find users older than 18",
    "Calculate average age",
    "Create friendship relations",
    "Count products by category"
]

print("Performance Test:")
print("="*70)

times = []
for i, query in enumerate(test_queries, 1):
    start = time.time()
    result, _ = engine.generate(query)
    elapsed = time.time() - start
    times.append(elapsed)
    
    print(f"\n{i}. {query}")
    print(f"   Time: {elapsed:.2f}s")
    print(f"   Result: {result[:80]}..." if len(result) > 80 else f"   Result: {result}")

print(f"\n{'='*70}")
print(f"Average time: {sum(times)/len(times):.2f}s")
print(f"Total time: {sum(times):.2f}s")

Performance Test:

1. Find all users
   Time: 0.70s
   Result: NHOMM_THEO_YY ?user 
LOCJ_RR(?user rdf:type :Person)

1. Find all users
   Time: 0.70s
   Result: NHOMM_THEO_YY ?user 
LOCJ_RR(?user rdf:type :Person)

2. Find users older than 18
   Time: 0.33s
   Result: LOCJ_RR(?age > 18)

2. Find users older than 18
   Time: 0.33s
   Result: LOCJ_RR(?age > 18)

3. Calculate average age
   Time: 0.31s
   Result: TRUNGG_BINH_PP(?age)

3. Calculate average age
   Time: 0.31s
   Result: TRUNGG_BINH_PP(?age)

4. Create friendship relations
   Time: 0.40s
   Result: XAY_DUNGWJ { ?p1 :friendOf ?p2 } NOII_MA_KK { ?p1 :livesIn ?city . ?p2 :livesIn ...

4. Create friendship relations
   Time: 0.40s
   Result: XAY_DUNGWJ { ?p1 :friendOf ?p2 } NOII_MA_KK { ?p1 :livesIn ?city . ?p2 :livesIn ...

5. Count products by category
   Time: 0.42s
   Result: NHOMM_THEO_YY ?category DEMM_JJ(?product) ?product :inCategory ?category

Average time: 0.43s
Total time: 2.16s

5. Count products by category
   Time:

## 8. Summary

### Key Improvements:
1. **Hierarchical Retrieval**: Classes → Properties (preserves context)
2. **Rich NL Patterns**: Leverages "Typical questions" in descriptions
3. **Type-Aware**: DSL entities categorized by kind (aggregate, filter, etc.)
4. **Scalable**: Handles 80 entities with nested properties
5. **Better Context**: Property descriptions include parent class

### Architecture:
- **OntologyParser**: Reads JSON-LD → Structured nodes
- **HierarchicalHyperGraph**: Domain classes with nested properties
- **DSLHyperGraph**: DSL entities with type/kind organization
- **Multi-Level Retrieval**: Class level + Property level + DSL level